# 0.2 模型的一生与路线设计 / The life of a model and the design of this roadmap

训练时和推理时是两条不同的时间线。弄清参数从哪里来、存在哪里，以及后面九个阶段为什么按这个顺序排。

Training time and inference time are two different timelines. See where parameters come from, where they live, and why the nine later stages come in this order.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 按部件统计 GPT-2 的参数量 / Count GPT-2's parameters by component

遍历 GPT-2 的参数，按名字把它们归入 token 嵌入、位置嵌入、注意力、MLP 和 LayerNorm 五类并统计数量。再写出总参数量的公式，并用它算出 GPT-2 small 和 GPT-2 medium 的参数量。

Walk over the parameters of GPT-2, sort them by name into token embedding, position embedding, attention, MLP and LayerNorm, and count each group. Then write the formula for the total and use it to compute the sizes of GPT-2 small and GPT-2 medium.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM


def count_by_component(model):
    """Where do GPT-2's parameters live?

    Returns a dict with integer counts for the keys
      "token_embedding", "position_embedding", "attention", "mlp", "layernorm", "total".
    Walk over model.transformer.named_parameters(); the names look like
      wte.weight, wpe.weight, h.0.ln_1.weight, h.0.attn.c_attn.weight, h.0.mlp.c_fc.bias, ln_f.weight
    """
    counts = {"token_embedding": 0, "position_embedding": 0, "attention": 0, "mlp": 0, "layernorm": 0}
    for name, p in model.transformer.named_parameters():
        # TODO 1: decide from `name` which bucket p.numel() belongs to
        raise NotImplementedError
    counts["total"] = sum(counts.values())
    return counts


def formula_count(V, n_ctx, d, L):
    """The same total from a closed-form formula.

    V vocabulary size, n_ctx maximum context length, d hidden size, L number of blocks.
    The MLP hidden size is 4d. The unembedding shares its weights with the token embedding.
    """
    # TODO 2: embeddings + L * (attention + mlp + two LayerNorms) + final LayerNorm
    raise NotImplementedError

In [ ]:
# 检查 / checks
from transformers import AutoModelForCausalLM

gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
c = count_by_component(gpt2)
expected = {"token_embedding": 38_597_376, "position_embedding": 786_432, "attention": 28_348_416,
            "mlp": 56_669_184, "layernorm": 38_400, "total": 124_439_808}
for k, v in expected.items():
    assert c[k] == v, f"{k}: got {c[k]:,}, expected {v:,}"
assert c["total"] == sum(p.numel() for p in gpt2.parameters()), "the unembedding shares the token embedding, so it adds nothing"
assert formula_count(50257, 1024, 768, 12) == 124_439_808, "the formula should give GPT-2 small"
assert formula_count(50257, 1024, 1024, 24) == 354_823_168, "and GPT-2 medium"
print("m02_params: all tests passed")
for k, v in c.items():
    print(f"  {k:20} {v:>12,}  {100 * v / c['total']:5.1f}%")

<details><summary><b>提示 1 / Hint 1</b></summary>

参数名里带 `.attn.` 的属于注意力，带 `.mlp.` 的属于 MLP，以 `wte` 和 `wpe` 开头的是两张嵌入表，其余都是 LayerNorm。

Names containing `.attn.` belong to attention and `.mlp.` to the MLP. Names starting with `wte` and `wpe` are the two embedding tables. Everything else is LayerNorm.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

公式照着上面的推导写即可。每层是 `4*d*d + 4*d` 加 `8*d*d + 5*d` 加 `4*d`，最后再加 `2*d`。

Follow the derivation above. Each layer is `4*d*d + 4*d` plus `8*d*d + 5*d` plus `4*d`, with `2*d` added at the end.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM


def count_by_component(model):
    counts = {"token_embedding": 0, "position_embedding": 0, "attention": 0, "mlp": 0, "layernorm": 0}
    for name, p in model.transformer.named_parameters():
        if name.startswith("wte"):
            key = "token_embedding"
        elif name.startswith("wpe"):
            key = "position_embedding"
        elif ".attn." in name:
            key = "attention"
        elif ".mlp." in name:
            key = "mlp"
        else:                                   # h.*.ln_1, h.*.ln_2, ln_f
            key = "layernorm"
        counts[key] += p.numel()
    counts["total"] = sum(counts.values())
    return counts


def formula_count(V, n_ctx, d, L):
    embeddings = V * d + n_ctx * d
    attention = (3 * d * d + 3 * d) + (d * d + d)          # Q, K, V in one matrix, then the output projection
    mlp = (d * 4 * d + 4 * d) + (4 * d * d + d)            # up to 4d, back down to d
    layernorms = 2 * (2 * d)                               # a scale and a shift, twice per block
    return embeddings + L * (attention + mlp + layernorms) + 2 * d
```

</details>

## 2. 概念自测 / Concept self-check

不看上面的内容，回答下面五问。
(a) 把这六步按发生的先后排序。softmax，分词，取最后一个位置的 logits，嵌入，选 token，通过 12 层 block。
(b) 下面哪些只发生在训练时？反向传播，计算交叉熵，采样，更新参数，分词。
(c) 输入有 5 个 token 时，GPT-2 small 输出的 logits 是什么形状？预测下一个 token 用的是哪一行？其余几行有什么用？
(d) 用 0.1 练习里的循环生成 20 个新 token，一共做了多少次前向计算？
(e) 第一步 A 的概率是 0.7，B 是 0.3。A 之后概率最高的 token 是 0.4，B 之后概率最高的 token 是 0.95。贪心解码得到的两步序列概率是多少？概率最大的两步序列是哪一条？

Without looking at the text above, answer these five questions.
(a) Put the six steps in the order they happen. Softmax, tokenization, taking the logits of the last position, embedding, choosing a token, passing through the 12 blocks.
(b) Which of these happen only during training? Backpropagation, computing cross-entropy, sampling, updating parameters, tokenization.
(c) With 5 input tokens, what is the shape of the logits GPT-2 small outputs? Which row predicts the next token? What are the other rows for?
(d) Generating 20 new tokens with the loop from the exercises of 0.1 takes how many forward passes?
(e) At step one A has probability 0.7 and B has 0.3. The most probable token after A has 0.4 and the most probable token after B has 0.95. What is the probability of the two-step sequence greedy decoding produces? Which two-step sequence is the most probable?

<details><summary><b>提示 1 / Hint 1</b></summary>

(c) 位置 $t$ 的 logits 预测的是第 $t+1$ 个 token。想一想训练时每个位置是否都有一个可以对照的真实 token。

(c) The logits at position $t$ predict token $t+1$. Consider whether, during training, every position has a true token to compare with.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(e) 序列的概率是两步概率的乘积，分别算出两条路径再比较。

(e) The probability of a sequence is the product of the two step probabilities. Compute both paths and compare.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 分词，嵌入，通过 12 层 block，取最后一个位置的 logits，softmax，选 token。

(b) 反向传播、计算交叉熵和更新参数只发生在训练时。采样只发生在推理时。分词在两边都有。

(c) 形状是 $(1, 5, 50257)$，每个输入位置一行。预测下一个 token 用最后一行。其余各行是模型在只看到前 $t$ 个 token 时对第 $t+1$ 个 token 的预测。训练时每一行都有真实的下一个 token 可以对照，所以一次前向就能得到 5 个训练信号。这是自回归模型训练效率高的原因之一。

(d) 20 次，每个新 token 一次，而且每次都把越来越长的整段输入重新算一遍。避免这种重复计算的办法是 KV cache，在 5.2 讲。

(e) 贪心第一步选 A，第二步选 A 之后最好的 token，概率为 $0.7 \times 0.4 = 0.28$。先选 B 的路径概率为 $0.3 \times 0.95 = 0.285$，略高于贪心的结果，所以概率最大的序列是从 B 开始的那一条。

---

(a) Tokenization, embedding, passing through the 12 blocks, taking the logits of the last position, softmax, choosing a token.

(b) Backpropagation, computing cross-entropy and updating parameters happen only during training. Sampling happens only at inference. Tokenization happens in both.

(c) The shape is $(1, 5, 50257)$, one row per input position. The last row predicts the next token. Each other row is the model's prediction of token $t+1$ after seeing only the first $t$ tokens. During training every row has a true next token to compare with, so one forward pass yields 5 training signals. This is one reason autoregressive training is efficient.

(d) 20, one per new token, and each one recomputes the whole, ever longer input. The KV cache removes this repeated work and is covered in 5.2.

(e) Greedy takes A first and then the best token after A, giving $0.7 \times 0.4 = 0.28$. The path that starts with B has $0.3 \times 0.95 = 0.285$, slightly above the greedy result, so the most probable sequence is the one that starts with B.

</details>